# 01 - Delta Lake com PySpark

Este notebook mostra operações ACID com Delta Lake.

## 1. Configuração da SparkSession com suporte Delta

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession

base = Path('..').resolve()
warehouse_dir = str((base / 'warehouse' / 'delta').resolve())

spark = (SparkSession.builder
 .appName('delta-lake-demo')
 .master('local[*]')
 .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
 .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
 .config('spark.sql.warehouse.dir', warehouse_dir)
 .getOrCreate())

print(warehouse_dir)

## 2. Leitura da fonte de dados

In [ ]:
df = spark.read.option('header', True).option('inferSchema', True).csv('../data/raw/vendas.csv')
df.printSchema()
df.show(5, truncate=False)
df.createOrReplaceTempView('vendas_csv')

## 3. Criação de banco e tabela Delta

In [ ]:
spark.sql('CREATE DATABASE IF NOT EXISTS delta_demo')
spark.sql('DROP TABLE IF EXISTS delta_demo.vendas')
spark.sql('''
CREATE TABLE delta_demo.vendas (
  venda_id INT,
  cliente_id INT,
  produto_id INT,
  quantidade INT,
  desconto DOUBLE
) USING DELTA
''')

spark.sql('''
INSERT INTO delta_demo.vendas
SELECT venda_id, cliente_id, produto_id, quantidade, desconto
FROM vendas_csv
''')

spark.sql('SELECT * FROM delta_demo.vendas ORDER BY venda_id').show()

## 4. INSERT de evidência

In [ ]:
spark.sql('INSERT INTO delta_demo.vendas VALUES (2001, 2, 104, 1, 0.0)')
spark.sql('SELECT * FROM delta_demo.vendas WHERE venda_id = 2001').show()

## 5. UPDATE de evidência

In [ ]:
spark.sql('UPDATE delta_demo.vendas SET desconto = 25.0 WHERE venda_id = 1002')
spark.sql('SELECT * FROM delta_demo.vendas WHERE venda_id = 1002').show()

## 6. DELETE de evidência

In [ ]:
spark.sql('DELETE FROM delta_demo.vendas WHERE venda_id = 1005')
spark.sql('SELECT * FROM delta_demo.vendas ORDER BY venda_id').show()